# Tier-1 Ridge ablation: standardization vs new features

**Intent:** tier 1 combined two changes (standardization + 4 new features + α CV) and showed a mild h/m regression at 3/5 snaps. Isolate which change is responsible before moving to tier 2.

**Four variants compared:**

| variant    | standardize | new features | α       |
|------------|-------------|--------------|---------|
| A `orig`   | no          | no (10)      | 10 fixed |
| B `std`    | **yes**     | no (10)      | CV       |
| C `feat`   | no          | **yes (14)** | CV       |
| D `t1`     | **yes**     | **yes (14)** | CV       |

- A = `ridge_orig` from `phase1_three_way.ipynb`.
- D = `ridge_t1` from `phase1_ridge_tier1.ipynb`.
- B and C are new here.

Comparisons that isolate each factor:
- **Standardization effect:** A vs B (same features), C vs D.
- **New-features effect:** A vs C (same std), B vs D.

**Plan doc:** `brainstorm/brainstorm_ridge_optimization.md`.


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

import _helpers as H

TIER1_CACHE = H.CACHE_DIR / 'phase1_ridge_tier1.pkl'
assert TIER1_CACHE.exists(), 'run phase1_ridge_tier1.ipynb first'

with open(TIER1_CACHE, 'rb') as f:
    df = pickle.load(f)
print(f'loaded {len(df)} rows  ·  columns: {len(df.columns)}')


## Configuration


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
BASE_FEATURES = [
    'observed_count', 'first_review_dbc', 'target_gap', 'observed_rate',
    'rate_last_day', 'rate_first_day', 'top_critic_frac',
    'pub_diversity', 'pub_entropy', 'low_activity_frac',
]
NEW_FEATURES = ['log_observed_count', 'log_rate_last_day', 'sqrt_rate_last_day', 'rate_delta']
ALL_FEATURES = BASE_FEATURES + NEW_FEATURES
ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
CV_FOLDS = 5
CV_SEED = 42


## Fit helpers

Per-snap α CV → then LOO prediction with best α. Two variants: with and without standardization.


In [ ]:
def select_alpha(X, y, standardize):
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=CV_SEED)
    best_alpha, best_mae = None, np.inf
    for alpha in ALPHA_GRID:
        fold_errs = []
        for train_idx, test_idx in kf.split(X):
            if standardize:
                model = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
            else:
                model = Ridge(alpha=alpha)
            model.fit(X[train_idx], y[train_idx])
            preds = model.predict(X[test_idx])
            fold_errs.extend(np.abs(preds - y[test_idx]).tolist())
        mae = float(np.mean(fold_errs))
        if mae < best_mae:
            best_mae, best_alpha = mae, alpha
    return best_alpha


def loo_predict(X, y, alpha, standardize):
    preds = np.zeros(len(X))
    for i in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[i] = False
        if standardize:
            model = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
        else:
            model = Ridge(alpha=alpha)
        model.fit(X[mask], y[mask])
        preds[i] = model.predict(X[i:i+1])[0]
    return preds


## Variant B: standardization + 10 base features (α CV)


In [ ]:
df['ridge_std_pred'] = np.nan
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days].dropna(subset=BASE_FEATURES + ['actual'])
    if len(sub) < CV_FOLDS * 2:
        continue
    X = sub[BASE_FEATURES].values
    y = sub['actual'].values.astype(float)
    best_alpha = select_alpha(X, y, standardize=True)
    preds = loo_predict(X, y, best_alpha, standardize=True)
    df.loc[sub.index, 'ridge_std_pred'] = preds
    print(f'  T-{snap_days}d (n={len(sub)}): α*={best_alpha}')


## Variant C: no standardization + 14 features (α CV)


In [ ]:
df['ridge_feat_pred'] = np.nan
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < CV_FOLDS * 2:
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    best_alpha = select_alpha(X, y, standardize=False)
    preds = loo_predict(X, y, best_alpha, standardize=False)
    df.loc[sub.index, 'ridge_feat_pred'] = preds
    print(f'  T-{snap_days}d (n={len(sub)}): α*={best_alpha}')


## Per-variant cohort summary


In [ ]:
def metrics(sub, pred_col):
    s = sub.dropna(subset=[pred_col])
    if len(s) == 0:
        return None
    err = s[pred_col].values - s['actual'].values
    return {
        'n': len(s), 'MAE': float(np.abs(err).mean()),
        'me': float(err.mean()),
    }


VARIANTS = [
    ('A orig',   'ridge_pred'),
    ('B std',    'ridge_std_pred'),
    ('C feat',   'ridge_feat_pred'),
    ('D t1',     'ridge_t1_pred'),
]

print('=== cohort MAE ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    print(f'T-{snap_days}d')
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            print(f'  {name:10s}  n={m["n"]:3d}  MAE={m["MAE"]:6.2f}  me={m["me"]:+6.2f}')
    print()


## h/m subset (n=2-5)


In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm_df = df[df['target_slug'].isin(HM)]

print('=== h/m MAE ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = hm_df[hm_df['snap_days'] == snap_days]
    if sub.empty:
        continue
    print(f'T-{snap_days}d  (n={len(sub)})')
    for name, col in VARIANTS:
        m = metrics(sub, col)
        if m:
            print(f'  {name:10s}  n={m["n"]:3d}  MAE={m["MAE"]:6.2f}  me={m["me"]:+6.2f}')
    print()


## Pairwise paired bootstrap (isolating each factor)

**Standardization effect:** A vs B, C vs D (same features).
**New-features effect:** A vs C, B vs D (same std state).


In [ ]:
COL_MAP = {name: col for name, col in VARIANTS}

PAIRS = [
    ('A orig',  'B std',   'std effect (no new)'),
    ('C feat',  'D t1',    'std effect (w/ new)'),
    ('A orig',  'C feat',  'new-feat effect (no std)'),
    ('B std',   'D t1',    'new-feat effect (w/ std)'),
]

print('=== pairwise bootstrap (ΔMAE: A − B, +ve → B wins) ===\n')
print(f'{"snap":<6}{"A":<10}{"B":<10}{"effect":<26}{"Δ units":>10}{"CI95_lo":>10}{"CI95_hi":>10}{"n":>6}  result')

for snap_days in SNAP_DAYS_LIST:
    snap_df = df[df['snap_days'] == snap_days]
    for a, b, label in PAIRS:
        a_col, b_col = COL_MAP[a], COL_MAP[b]
        paired = snap_df.dropna(subset=[a_col, b_col, 'actual'])
        if len(paired) < 5:
            continue
        a_abs = np.abs(paired[a_col].values - paired['actual'].values)
        b_abs = np.abs(paired[b_col].values - paired['actual'].values)
        deltas = a_abs - b_abs
        point, lo, hi = H.bootstrap_mae_delta(deltas, n_boot=1000)
        if lo > 0:
            result = f'{b:>10} wins'
        elif hi < 0:
            result = f'{a:>10} wins'
        else:
            result = '    ns'
        print(f'T-{snap_days}d  {a:<10}{b:<10}{label:<26}{point:>+10.3f}{lo:>+10.3f}{hi:>+10.3f}'
              f'{len(paired):>6}  {result}')
    print()


## h/m-only pairwise comparison

Small n but let's see signed direction at least (bootstrap CI will usually be ns on n=2-5).


In [ ]:
print('=== h/m pairwise delta-MAE (signed only, no CI) ===\n')
print(f'{"snap":<6}{"A":<10}{"B":<10}{"effect":<26}{"delta (A-B)":>14}{"n":>6}')
for snap_days in SNAP_DAYS_LIST:
    snap_df = hm_df[hm_df['snap_days'] == snap_days]
    for a, b, label in PAIRS:
        a_col, b_col = COL_MAP[a], COL_MAP[b]
        paired = snap_df.dropna(subset=[a_col, b_col, 'actual'])
        if len(paired) < 1:
            continue
        a_abs = np.abs(paired[a_col].values - paired['actual'].values)
        b_abs = np.abs(paired[b_col].values - paired['actual'].values)
        delta = (a_abs - b_abs).mean()  # >0 means B wins
        print(f'T-{snap_days}d  {a:<10}{b:<10}{label:<26}{delta:>+14.3f}{len(paired):>6}')
    print()

## Observations

*(fill in after run)*
